# Silver Exploration : Weather


In [1]:
import os, findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract, col

spark = SparkSession.builder.appName("silver-exploration-weather").getOrCreate()

RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 11:34:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/10 11:34:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Exploration of the weather dataset

The weather data comes from Open-Meteo as one JSON file per city and download. Each file becomes *one row* in Spark, so the DataFrame has one row per file, not per hour.

- The hourly measurements are stored as **parallel arrays** inside the `hourly` struct.
- The metadata is stored once per file.
- The timezone is `GMT` and `utc_offset_seconds` is 0, so all timestamps are in **UTC**. This matches the trip timestamps.
- The files offer 10 hourly variables. For the analysis the following have been selected: `temperature_2m`, `apparent_temperature`, `rain`, `precipitation`, and `wind_speed_10m`.

In [2]:
weather = (spark.read
           .option("multiLine", True)
           .json(f"{RAW}/weather/*/*.json"))

weather.printSchema()

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- apparent_temperature: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- cloud_cover: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- precipitation: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- rain: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- snowfall: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- weather_code: array (nullable = true)
 |    |    |-- el

### Turning Arrays into Rows

Each file stores the hours as arrays, but they should be in **one row per hour**.

- `arrays_zip` combines the arrays element by element, so the values for the same hour stay together.
- `explode` turns each zipped element into its own row.
- The file path is kept as `src`, to be able to find the city and the download time.

In [3]:
from pyspark.sql.functions import arrays_zip, explode

w = (weather
     .withColumn("src", input_file_name())
     .select("src", explode(arrays_zip(
         col("hourly.time"),
         col("hourly.temperature_2m"),
         col("hourly.rain"),
         col("hourly.wind_speed_10m"),
        col("hourly.apparent_temperature"),
        col("hourly.precipitation")
     )).alias("h")))

w.select("src", "h.*").show(3, truncate = False)

+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|src                                                                                                                    |time            |temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+-----------------------------------------------------------------------------------------------------------------------+----------------+--------------+----+--------------+--------------------+-------------+
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T00:00|0.5           |0.0 |7.7           |-3.0                |0.0          |
|file:///home/ubuntu/spark-lab-data/dev/project/raw/weather/trondheim/2022-01-01_2026-10-18_20261019T112729Z-52a0eb.json|2022-01-01T01:00|0.2           |0.0 |7.6   

### Weather Analysis

In [4]:
from pyspark.sql.functions import input_file_name, regexp_extract, element_at, size

(weather
 .select(
     regexp_extract(input_file_name(), r"/weather/([^/]+)/(.+)\.json", 1).alias("city"),
     regexp_extract(input_file_name(), r"/weather/([^/]+)/(.+)\.json", 2).alias("file"),
     element_at("hourly.time", 1).alias("first_hour"),
     element_at("hourly.time", -1).alias("last_hour"),
     size("hourly.time").alias("hours"))
 .orderBy("city", "first_hour")
 .show(truncate=False))

+------+---------------------------------------------+----------------+----------------+-----+
|city  |file                                         |first_hour      |last_hour       |hours|
+------+---------------------------------------------+----------------+----------------+-----+
|bergen|2022-01-01_2026-10-18_20261019T112729Z-52a0eb|2022-01-01T00:00|2026-10-18T23:00|42048|
|bergen|2026-10-06_2026-10-19_20261020T140223Z-00e429|2026-10-06T00:00|2026-10-19T23:00|336  |
|bergen|2026-10-09_2026-10-22_20261023T130145Z-640073|2026-10-09T00:00|2026-10-22T23:00|336  |
|bergen|2026-10-09_2026-10-22_20261023T132139Z-5e2ca9|2026-10-09T00:00|2026-10-22T23:00|336  |
|bergen|2026-10-19_2026-11-01_20261102T111527Z-8a2ef2|2026-10-19T00:00|2026-11-01T23:00|336  |
|bergen|2026-10-19_2026-11-01_20261102T111718Z-0a8ea7|2026-10-19T00:00|2026-11-01T23:00|336  |
|bergen|2026-10-27_2026-11-09_20261110T104702Z-0c9509|2026-10-27T00:00|2026-11-09T23:00|336  |
|bergen|2026-10-27_2026-11-09_20261110T104739Z-560

#### Overlap Between Downloads

Each row above is one downloaded file. For every city there is one large backfill file that starts on 2022-01-01, followed by smaller top-up files that start in October 2026. The top-up files cover about 14 days each, and those days are also inside the backfill file (or an earlier top-up). The same hours therefore exist in several files, so we have to deal with duplicates in Silver.

#### City and Duplicate Check

The files only contain latitude and longitude values of the cities, not the city name itself, so the name is taken from the file path (`/weather/<city>/`) with `regexp_extract`.

- The first number in the output is the **total number of weather rows**.
- The second is the number of **distinct (city, hour) pairs**.

In [5]:
from pyspark.sql.functions import regexp_extract

wx = (w.select("src", "h.*")
      .withColumn("city", regexp_extract("src", r"/weather/([^/]+)/", 1)))

print(wx.count())
print(wx.select("city", "time").distinct().count())

135216
127728


There are 135,216 rows but only 127,728 distinct (city, hour) pairs, which means there are 7,488 rows that are repeated from an already existing hour. The duplicates come from the 14-day overlap in the top-up downloads. The Bronze layer is in charge of downloading the data, and the Silver layer handles the cleaning where duplicate files get handled. **The rule is to keep the most recent data.**

In [6]:
from pyspark.sql.functions import count, countDistinct

dups = (wx.groupBy("city", "time")
        .agg(count("*").alias("n"),
             countDistinct("temperature_2m").alias("temps"),
             countDistinct("precipitation").alias("precs"))
        .filter(col("n") > 1))

print(dups.count())
print(dups.filter((col("temps") > 1) | (col("precs") > 1)).count())

2520


0


#### Do the Repeated Hours Contain Different Values?

Group by city and hour and count how many versions of each hour exist, and the different temperature and precipitation values that are present.

- 2,520 (city, hour) pairs appear more than once. Some appear three or more times, which explains the surplus of rows compared to the surplus of pairs.
- **None** of them have different temperature or precipitation values between copies.

The re-fetched data is identical, and it does not matter which data is kept. Still choose a deterministic rule: **keep the copy from the newest download**. Should Open-Meteo make changes, the revised value will be chosen.

##### Check if any weather values are missing

In [7]:
from pyspark.sql.functions import sum as spark_sum, when

weather_cols = ["temperature_2m", "rain", "wind_speed_10m",
                "apparent_temperature", "precipitation"]

wx.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in weather_cols
]).show()

+--------------+----+--------------+--------------------+-------------+
|temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|
+--------------+----+--------------+--------------------+-------------+
|             0|   0|             0|                   0|            0|
+--------------+----+--------------+--------------------+-------------+



All five weather columns have **0 missing values**, so there is no need to fill in or remove any hours for missing data.

#### Removing duplicates with a window function

- `run_id` is read from the file name. It is the time of the download plus a short code, so a newer download has a larger `run_id`.
- A window groups the rows by **city and hour**. Within each group, they are ordered by `run_id`, with the newest first.
- `row_number()` numbers the rows in each group. Keep row number 1, which is the newest copy, and drop the helper columns.

The partition key (`city`, `time`) decides what counts as "the same observation". If partitioning by file occurred, each hour would be unique within its own file and it would appear as if there were no duplicates.

In [8]:
from pyspark.sql.functions import to_timestamp, regexp_extract, row_number, desc, col
from pyspark.sql.window import Window

wx2 = (wx
       .withColumn("ts_utc", to_timestamp("time", "yyyy-MM-dd'T'HH:mm"))
       .withColumn("run_id", regexp_extract("src", r"_([0-9T]+Z-[0-9a-f]+)\.json", 1)))

win = Window.partitionBy("city", "time").orderBy(desc("run_id"))

silver_weather = (wx2
                  .withColumn("rn", row_number().over(win))
                  .filter(col("rn") == 1)
                  .drop("rn", "src", "time"))

print(silver_weather.count())
silver_weather.show(3)

127728


+--------------+----+--------------+--------------------+-------------+------+-------------------+--------------------+
|temperature_2m|rain|wind_speed_10m|apparent_temperature|precipitation|  city|             ts_utc|              run_id|
+--------------+----+--------------+--------------------+-------------+------+-------------------+--------------------+
|           2.0| 0.0|           5.1|                -0.6|          0.0|bergen|2022-01-01 03:00:00|20261019T112729Z-...|
|           3.7| 0.0|          14.0|                -0.0|          0.0|bergen|2022-01-01 10:00:00|20261019T112729Z-...|
|           4.9| 0.3|          17.3|                 0.6|          0.3|bergen|2022-01-01 13:00:00|20261019T112729Z-...|
+--------------+----+--------------+--------------------+-------------+------+-------------------+--------------------+
only showing top 3 rows



In [9]:
from pyspark.sql.functions import min, max, count, countDistinct

silver_weather.groupBy("city").agg(
    count("*").alias("rows"),
    countDistinct("ts_utc").alias("distinct_hours"),
    min("ts_utc").alias("first_hour"),
    max("ts_utc").alias("last_hour")
).show(truncate=False)

+---------+-----+--------------+-------------------+-------------------+
|city     |rows |distinct_hours|first_hour         |last_hour          |
+---------+-----+--------------+-------------------+-------------------+
|trondheim|42576|42576         |2022-01-01 00:00:00|2026-11-09 23:00:00|
|oslo     |42576|42576         |2022-01-01 00:00:00|2026-11-09 23:00:00|
|bergen   |42576|42576         |2022-01-01 00:00:00|2026-11-09 23:00:00|
+---------+-----+--------------+-------------------+-------------------+



#### Checking the Result per City

For each city, the number of rows, distinct hours, and the first and last hour in the city gets compared.

- Rows equal distinct hours for all three cities, so there is exactly one row per city and hour.
- The time range is 2022-01-01 00:00 to 2026-11-09 23:00. That is 1,774 days $\times$ 24 $=$ 42,576 hours, the same as the row count, so there are **no gaps** either.
- All three cities have the same number of rows, so the join with trips later will not lose a city.

#### Moving the Weather Logic into a Module

The code above is now in `Project/Silver/silver_weather.py` as `read_raw_weather`, `build_silver_weather` and `weather_quality_metrics`, so the pipeline can run it without the notebook. The cells below import it and check that it gives the same results as the exploration. The module also adds a `_silver_processed_time` column to show when each row was produced.

In [10]:
import os
print(os.getcwd())
print(os.path.exists("../Silver/silver_weather.py"))

/home/ubuntu/group11/Project/notebooks
True


In [11]:
import sys
sys.path.append("/home/ubuntu/group11/Project/Silver")
import silver_weather

In [12]:
import importlib

importlib.reload(silver_weather)
print(silver_weather.weather_quality_metrics(spark, RAW))
sw = silver_weather.build_silver_weather(spark, RAW)
print(sw.count())
sw.select("city", "ts_utc", "_silver_processed_time").show(3, truncate=False)
print(sw.filter(col("ts_utc").isNull()).count())

{'layer': 'silver_weather', 'rows_in': 135216, 'rows_out': 127728, 'duplicates_removed': 7488}
127728


+------+-------------------+-----------------------+
|city  |ts_utc             |_silver_processed_time |
+------+-------------------+-----------------------+
|bergen|2022-01-01 01:00:00|2026-11-10 11:34:43.398|
|bergen|2022-01-01 03:00:00|2026-11-10 11:34:43.398|
|bergen|2022-01-01 05:00:00|2026-11-10 11:34:43.398|
+------+-------------------+-----------------------+
only showing top 3 rows

0


Quality metrics for the weather layer:

- rows in (Bronze): 135,216
- rows out (Silver): 127,728
- duplicates removed: 7,488

The count of removed duplicates grows with every scheduled run, because each run adds another 14-day overlap. The Silver table itself stays at one row per city and hour. The count of null timestamps is 0, so every time string was parsed.

In [13]:
from pyspark.sql.functions import min as smin, max as smax, countDistinct, count

sw.groupBy("city").agg(count("*").alias("rows"),
         countDistinct("ts_utc").alias("distinct_hours"),
        smin("ts_utc").alias("first"),
        smax("ts_utc").alias("last")).show(truncate=False)

+---------+-----+--------------+-------------------+-------------------+
|city     |rows |distinct_hours|first              |last               |
+---------+-----+--------------+-------------------+-------------------+
|trondheim|42576|42576         |2022-01-01 00:00:00|2026-11-09 23:00:00|
|oslo     |42576|42576         |2022-01-01 00:00:00|2026-11-09 23:00:00|
|bergen   |42576|42576         |2022-01-01 00:00:00|2026-11-09 23:00:00|
+---------+-----+--------------+-------------------+-------------------+



The module gives one row per city and hour, with the same number of hours for each city (42,576), now up to 2026-11-09. This is the same result as the exploration, with newer data.